# 14.2 高级提示技术 (Advanced Prompting)

高级提示技术通过更复杂的提示策略进一步提升模型性能。

本节涵盖：自我一致性、ReAct、方向性刺激提示

## 1. 自我一致性与ReAct

**自我一致性（Self-Consistency）**：对同一问题多次采样CoT推理路径，取多数投票作为最终答案。

**ReAct**：结合推理（Reasoning）和行动（Acting），模型交替进行思考和工具调用。

**ReAct流程**：
1. Thought: 思考下一步
2. Action: 调用工具
3. Observation: 观察结果
4. 重复直到得出答案

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter

torch.manual_seed(42)

class SelfConsistencyDemo(nn.Module):
    def __init__(self, d=64, n_answers=5):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, 32), nn.ReLU(), nn.Linear(32, n_answers))

    def sample_answers(self, query, n_samples=10, temperature=0.7):
        logits = self.net(query) / temperature
        answers = []
        for _ in range(n_samples):
            probs = F.softmax(logits + torch.randn_like(logits) * 0.5, dim=-1)
            answer = probs.argmax(dim=-1).item()
            answers.append(answer)
        return answers

model = SelfConsistencyDemo()
query = torch.randn(1, 64)
answers = model.sample_answers(query, n_samples=20)
vote_counts = Counter(answers)
consensus = vote_counts.most_common(1)[0]

print('=== Self-Consistency ===')
print(f'Sampled answers: {answers}')
print(f'Vote counts: {dict(vote_counts)}')
print(f'Consensus answer: {consensus[0]} (votes: {consensus[1]}/{len(answers)})')
print(f'\nKey: More samples -> more reliable consensus.')
print(f'\n=== ReAct Pattern ===')
print(f'Thought: I need to find the population of Paris.')
print(f'Action: search("Paris population")')
print(f'Observation: Paris has ~2.1M inhabitants.')
print(f'Thought: I have the answer.')
print(f'Answer: Paris has approximately 2.1 million inhabitants.')
print(f'\nKey: ReAct interleaves reasoning with tool use for complex tasks.')